In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from utils import ChocolateCountDataset, ChocolateCounterNetDeep, ChocolateCounterFasterRCNN
from tqdm import tqdm
import os
import pandas as pd

from sklearn.model_selection import train_test_split


In [2]:
import os

all_filenames = sorted([f for f in os.listdir("../dataset_project_iapr2025/train") if f.endswith(('.JPG'))])

train_filenames, val_filenames = train_test_split(all_filenames[13:], test_size=0.33, random_state=42)

train_filenames += all_filenames[:13]

In [5]:

# Paths (adjust as needed)
TRAIN_IMAGE_DIR = "../dataset_project_iapr2025/train"
VAL_IMAGE_DIR = "../dataset_project_iapr2025/train"
LABEL_DIR = "./labels"

# Hyperparameters
BATCH_SIZE = 32
NUM_EPOCHS = 400
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_PATH = "best_model.pth"

# Dataset and loaders
train_dataset = ChocolateCountDataset(
    image_filenames=train_filenames,
    image_dir=TRAIN_IMAGE_DIR,
    label_dir=LABEL_DIR
)
val_dataset = ChocolateCountDataset(
    image_filenames=val_filenames,
    image_dir=VAL_IMAGE_DIR,
    label_dir=LABEL_DIR
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# Model
model = ChocolateCounterFasterRCNN().to(DEVICE)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=LEARNING_RATE, weight_decay=1e-4)

# Training loop
def evaluate(model, loader):
    model.train()  # must be in train mode to compute loss, Faster-RCNN specificity
    total_loss = 0
    with torch.no_grad():
        for images, targets in loader:
            images = list(img.to(DEVICE) for img in images)
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            total_loss += losses.item() * len(images)
    return total_loss / len(loader.dataset)

patience = 15
epochs_no_improve = 0
best_val_loss = float("inf")


for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0
    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        images = list(img.to(DEVICE) for img in images)
        # print(targets)
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running_loss += losses.item() * len(images)

    avg_train_loss = running_loss / len(train_loader.dataset)
    val_loss = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

Epoch 1/400: 100%|██████████| 3/3 [00:23<00:00,  7.95s/it]


Epoch 1: Train Loss = 3.3884, Val Loss = 3.4859


Epoch 2/400: 100%|██████████| 3/3 [00:22<00:00,  7.43s/it]


Epoch 2: Train Loss = 3.4595, Val Loss = 3.4296


Epoch 3/400: 100%|██████████| 3/3 [00:23<00:00,  7.90s/it]


Epoch 3: Train Loss = 3.3575, Val Loss = 3.2203


... (78 outputs left out)

Epoch 43/400: 100%|██████████| 3/3 [00:21<00:00,  7.08s/it]


Epoch 43: Train Loss = 0.9165, Val Loss = 1.0761


Epoch 44/400: 100%|██████████| 3/3 [00:21<00:00,  7.24s/it]


Epoch 44: Train Loss = 0.9445, Val Loss = 1.0964
Early stopping at epoch 44


In [8]:
import os
import torch
import pandas as pd
from torchvision import transforms
from torch.utils.data import DataLoader
from collections import Counter
from PIL import Image


TEST_IMAGE_DIR = "../dataset_project_iapr2025/test"
MODEL_CHECKPOINT = "best_model.pth"
OUTPUT_CSV_PATH = "submission.csv"
CLASS_NAMES_CSV = [
    "Jelly White", "Jelly Milk", "Jelly Black", "Amandina", "Crème brulée",
    "Triangolo", "Tentation noir", "Comtesse", "Noblesse", "Noir authentique",
    "Passion au lait", "Arabia", "Stracciatella"
]
NUM_CLASSES = len(CLASS_NAMES_CSV)

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ChocolateCounterFasterRCNN(num_classes=NUM_CLASSES).to(device)
model.load_state_dict(torch.load(MODEL_CHECKPOINT, map_location=device))
model.eval()

# Test dataset
test_filenames = sorted([f for f in os.listdir("../dataset_project_iapr2025/test") if f.endswith(('.JPG'))])
transform = transforms.Compose([
    transforms.Resize((384, 576)),
    transforms.ToTensor()
])

# Predict and collect results
results = []

with torch.no_grad():
    for fname in test_filenames:
        img_path = os.path.join(TEST_IMAGE_DIR, fname)
        image = Image.open(img_path).convert("RGB")
        image_tensor = transform(image).to(device).unsqueeze(0)  # shape [1, C, H, W]

        predictions = model(image_tensor)[0]
        labels = predictions["labels"].cpu().tolist()

        # Count per class (labels are 1-based)
        class_counts = Counter(labels)
        counts = [class_counts.get(i + 1, 0) for i in range(NUM_CLASSES)]

        image_id = os.path.splitext(fname)[0].replace("IMG_", "").replace("L", "")
        results.append([image_id] + counts)

# Save to CSV
df = pd.DataFrame(results, columns=["id"] + CLASS_NAMES_CSV)
df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Submission file saved to {OUTPUT_CSV_PATH}")

/tmp/ipykernel_324371/514352915.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_CHECKPOINT, map_location=device))


Submission file saved to submission.csv
